# 04b – Modeling v2 (Features aus 03b)

Vergleich: **`04_modeling.ipynb`** (v1-Parquets) · Features: [10_PREPROCESSING_V2.md](../docs/10_PREPROCESSING_V2.md)

Input: `train_labeled_v2.parquet`, `test_features_v2.parquet` (nach **03b**)

Output: `submission_{mode}_v2.csv` — optional Blend mit Persistence (35 %).

**RAM (Colab full):** Parquet → `/content/`, sofort **täglich→wöchentlich**, Tages-`train_df` löschen, Test auf eine Zeile/Region reduzieren. **Parallel:** `DM_WORKERS` (Colab-Default: 1).

In [ ]:
import gc
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

RANDOM_STATE = 42
VAL_REGION_FRAC = 0.2
BLEND_PERSIST = 0.35  # 0 = nur Modell; höher = näher an Baseline 1

_repo = Path("/content/DataMining_Final-Project")
if not (_repo / "scripts" / "notebook_init.py").exists():
    try:
        import google.colab  # noqa: F401
        import subprocess
        subprocess.run(
            ["git", "clone", "--branch", "main",
             "https://github.com/jspldrch/DataMining_Final-Project.git", str(_repo)],
            check=True,
        )
    except ImportError:
        pass
for _p in (Path.cwd(), Path.cwd().parent, _repo):
    if (_p / "scripts" / "notebook_init.py").exists() and str(_p.resolve()) not in sys.path:
        sys.path.insert(0, str(_p.resolve()))

from scripts.notebook_init import setup
from scripts.project_env import load_script, read_parquet_notebook
from scripts.parallel_util import default_workers

env = setup()
N_WORKERS = default_workers()
print(f"Parallel workers: {N_WORKERS}  (DM_WORKERS=1 zum Abschalten)")
PROJECT_ROOT = env["project_root"]
DATA_DIR = env["data_dir"]
PROCESSED_DIR = env["processed_dir"]
SUBMISSION_DIR = env["submission_dir"]
MODE = env["mode"]

_fv2 = load_script("features_v2", PROJECT_ROOT / "scripts" / "features_v2.py")
feature_columns_v2 = _fv2.feature_columns_v2
_wm = load_script("weekly_model", PROJECT_ROOT / "scripts" / "weekly_model.py")

train_df = read_parquet_notebook(PROCESSED_DIR / "train_labeled_v2.parquet")
FEATURES = [c for c in feature_columns_v2() if c in train_df.columns]
train_df = _wm.slim_for_modeling(train_df, FEATURES)
n_daily = len(train_df)
train_weekly = _wm.daily_to_weekly(train_df)
del train_df
gc.collect()

wk = _wm.weekly_summary(train_weekly, already_weekly=True, daily_rows=n_daily)
TRAIN_SCORE_MEDIAN = float(train_weekly["score"].median())
print(f"Features v2: {len(FEATURES)} | {wk['daily_rows']:,} Tage → {wk['weekly_rows']:,} Wochen")

test_df = read_parquet_notebook(PROCESSED_DIR / "test_features_v2.parquet")
X_test = _wm.test_features_last_row(test_df, FEATURES)
del test_df
gc.collect()

SAMPLE_SUB_PATH = _wm.find_sample_submission(PROJECT_ROOT, DATA_DIR)
SUBMISSION_TEMPLATE = (
    SAMPLE_SUB_PATH if SAMPLE_SUB_PATH is not None
    else _wm.build_submission_template(X_test["region_id"])
)
print(f"Template: {SAMPLE_SUB_PATH or 'aus test_features'}")

## 1. Train/Valid

In [ ]:
X_tr, y_tr, X_va, y_va, val_regions = _wm.build_region_holdout(
    train_weekly,
    FEATURES,
    val_region_frac=VAL_REGION_FRAC,
    seed=RANDOM_STATE,
    n_workers=N_WORKERS,
    already_weekly=True,
)
print(f"Train-Fenster: {len(X_tr):,} | Val-Regionen: {len(val_regions)}")

## 2. Baselines

In [ ]:
last_score = train_weekly.sort_values("ordinal").groupby("region_id")["score"].last()
region_median = train_weekly.groupby("region_id")["score"].median()

def eval_preds(y_true, y_pred, name):
    print(f"{name:32s}  MAE={_wm.mae_kaggle(y_true, y_pred):.4f}")

persist = np.column_stack([last_score.reindex(val_regions).fillna(0).to_numpy() for _ in range(5)])
eval_preds(y_va, persist, "Persistence (letzter Score ×5)")
med = np.column_stack([
    region_median.reindex(val_regions).fillna(TRAIN_SCORE_MEDIAN).to_numpy() for _ in range(5)
])
eval_preds(y_va, med, "Regional median ×5")

## 3. LightGBM

In [ ]:
# Normal import (nach git pull); load_script nur Fallback
try:
    from scripts.modeling_train import train_week_models_parallel
except ImportError:
    _mt = load_script("modeling_train", PROJECT_ROOT / "scripts" / "modeling_train.py")
    train_week_models_parallel = _mt.train_week_models_parallel
cat_feats = ["region_id"] if "region_id" in FEATURES else None

lgb_params = dict(
    objective="regression",
    metric="mae",
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=63,
    min_child_samples=60,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    verbose=-1,
)

print(f"LightGBM: 5 Wochen parallel (max {min(N_WORKERS, 5)} Prozesse) …")
models = train_week_models_parallel(
    X_tr, y_tr, X_va, y_va, lgb_params,
    categorical_feature=cat_feats,
    early_stopping_rounds=50,
    n_workers=min(N_WORKERS, 5),
)
val_preds = np.column_stack([_wm.clip_scores(m.predict(X_va)) for m in models])
eval_preds(y_va, val_preds, "LightGBM v2 (5 Modelle, parallel)")

## 4. Submission (+ optional Persistence-Blend)

In [ ]:
del X_tr, y_tr, X_va, y_va
gc.collect()

X_all, y_all, _ = _wm.build_sliding_samples(
    train_weekly,
    FEATURES,
    already_weekly=True,
    n_workers=N_WORKERS,
)
final_models = []
for week in range(5):
    p = dict(lgb_params)
    p["n_estimators"] = getattr(models[week], "best_iteration_", None) or 800
    p["random_state"] = RANDOM_STATE + week
    p["n_jobs"] = -1
    m = lgb.LGBMRegressor(**p)
    fit_kw = {}
    if cat_feats:
        fit_kw["categorical_feature"] = cat_feats
    m.fit(X_all, y_all[:, week], **fit_kw)
    final_models.append(m)
print("Finales Training: 5 Modelle (je best_iteration aus Validierung)")

del X_all, y_all
gc.collect()

test_preds = _wm.predict_week_columns(final_models, X_test)

if BLEND_PERSIST > 0:
    last_s = last_score.reindex(X_test["region_id"].astype(str)).fillna(0).to_numpy(dtype=float)
    persist = np.column_stack([last_s for _ in range(5)])
    test_preds = _wm.clip_scores((1 - BLEND_PERSIST) * test_preds + BLEND_PERSIST * persist)
    print(f"Blend: {(1-BLEND_PERSIST):.0%} Modell + {BLEND_PERSIST:.0%} Persistence")

submission = _wm.submission_frame(X_test["region_id"], test_preds)
submission = _wm.align_to_sample_submission(submission, SUBMISSION_TEMPLATE)

out_path = SUBMISSION_DIR / f"submission_{MODE}_v2.csv"
submission.to_csv(out_path, index=False)
print(f"Gespeichert: {out_path} ({len(submission):,} Zeilen)")
submission.head(10)